## Lab 03
### Luis Gonzalez

In [1]:
from spark_utils import SparkUtils
from pyspark.sql.functions import col, when, count, avg

spark_url = "spark://spark-master:7077"
app_name = "Spark SQL - Lab03"
su = SparkUtils(spark_url, app_name)
spark = su._spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 01:57:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
"""
columns_types = [("Order_ID", "long"),
                 ("Company", "string"),
                 ("City", "string"),
                 ("Customer_Age", "int"),
                 ("Order_Value", "float"),
                 ("Delivery_Time_Min", "float"),
                 ("Distance_Km", "float"),
                 ("Items_Count", "float"),
                 ("Product_Category", "string"),
                 ("Payment_Method", "string"),
                 ("Customer_Rating", "float"),
                 ("Discount_Applied", "float"),
                 ("Delivery_Partner_Rating", "float")
                 ]

qcommerce_schema = SparkUtils.generate_schema(columns_types)

qcommerce_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(qcommerce_schema) \
                .csv("/opt/spark/work-dir/data/")

qcommerce_df.printSchema()
qcommerce_df.show(5)

root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)



[Stage 0:>                                                          (0 + 1) / 1]

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.0|                    3.2|
| 1000003|Flipkart Minute

In [3]:
qcommerce_df = qcommerce_df.withColumn(
    "Delivery_SLA",
    when(col("Delivery_Time_Min") <= 15, "FAST")
    .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON_TIME")
    .otherwise("LATE")
)

qcommerce_df \
    .filter(col("Delivery_SLA") == "LATE") \
    .orderBy(col("Delivery_Time_Min").desc()) \
    .select(
        "Order_ID",
        "Company",
        "City",
        "Delivery_Time_Min",
        "Delivery_SLA"
    ) \
    .show()

[Stage 1:=============================>                             (1 + 1) / 2]

+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1529779|Jio Mart|Haridwar|             40.0|        LATE|
| 1807126|Jio Mart|Haridwar|             40.0|        LATE|
| 1165764|Jio Mart|Haridwar|           39.994|        LATE|
| 1610720|Jio Mart|Haridwar|           39.994|        LATE|
| 1729503|Jio Mart|Haridwar|           39.994|        LATE|
| 1951122|Jio Mart|Haridwar|           39.988|        LATE|
| 1975896|Jio Mart|Haridwar|           39.988|        LATE|
| 1059671|Jio Mart|Haridwar|           39.982|        LATE|
| 1594835|Jio Mart|Haridwar|           39.982|        LATE|
| 1162804|Jio Mart|Haridwar|           39.982|        LATE|
| 1826295|Jio Mart|Haridwar|           39.982|        LATE|
| 1961544|Jio Mart|Haridwar|           39.976|        LATE|
| 1360875|Jio Mart|Haridwar|           39.964|        LATE|
| 1555058|Jio Mart|Haridwar|           3

In [4]:
qcommerce_df = qcommerce_df.withColumn(
    "Effective_Order_Value",
    col("Order_Value") * (1 - col("Discount_Applied"))
)

qcommerce_df = qcommerce_df.withColumn(
    "Price_Bucket",
    when(col("Effective_Order_Value") < 200, "LOW")
    .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500), "MEDIUM")
    .otherwise("HIGH")
)

qcommerce_df \
    .groupBy("Price_Bucket") \
    .agg(
        count("*").alias("orders"),
        avg("Effective_Order_Value").alias("avg_effective_value")
    ) \
    .orderBy(col("avg_effective_value").desc()) \
    .show()

[Stage 2:=============================>                             (1 + 1) / 2]

+------------+------+-------------------+
|Price_Bucket|orders|avg_effective_value|
+------------+------+-------------------+
|        HIGH|268953|  740.4337238893675|
|      MEDIUM|210429|  358.0973233400432|
|         LOW|520618| 21.591204157544553|
+------------+------+-------------------+



In [5]:
qcommerce_df = qcommerce_df.filter(
    (col("Customer_Age").isNotNull()) &
    (col("Customer_Age") >= 0) &
    (col("Customer_Age") <= 100)
)

qcommerce_df = qcommerce_df.withColumn(
    "Age_Group",
    when(col("Customer_Age") < 25, "YOUNG")
    .when((col("Customer_Age") >= 25) & (col("Customer_Age") <= 44), "ADULT")
    .otherwise("SENIOR")
)

qcommerce_df \
    .groupBy("Age_Group", "Product_Category") \
    .agg(
        count("*").alias("orders"),
        avg("Order_Value").alias("avg_order_value")
    ) \
    .orderBy(
        col("Age_Group").asc(),
        col("orders").desc()
    ) \
    .show()

[Stage 5:=============================>                             (1 + 1) / 2]

+---------+-------------------+------+-----------------+
|Age_Group|   Product_Category|orders|  avg_order_value|
+---------+-------------------+------+-----------------+
|    ADULT|              Dairy| 68512|573.4268899414931|
|    ADULT|      Personal Care| 68331| 569.512671998805|
|    ADULT|          Groceries| 68291|571.5250993844182|
|    ADULT|          Household| 68110|572.9259149188181|
|    ADULT|             Snacks| 68100|570.3797974095505|
|    ADULT|          Beverages| 67979|572.0329877095578|
|    ADULT|Fruits & Vegetables| 67736|569.8593599596651|
|   SENIOR|          Groceries| 51030|572.2596391052539|
|   SENIOR|              Dairy| 51025| 573.781117268945|
|   SENIOR|             Snacks| 50924| 572.687851608881|
|   SENIOR|          Household| 50803| 571.172082385334|
|   SENIOR|          Beverages| 50746| 568.141013231256|
|   SENIOR|      Personal Care| 50707|571.1642560109325|
|   SENIOR|Fruits & Vegetables| 50642|573.7244422534909|
|    YOUNG|              Dairy|